# Create legal hard-negative triplets

This notebook reads the `legal` table, uses the embedding model to find one hard negative for each query, and writes the result to `/kaggle/working/legal_triplet.db`.

Only the first `MAX_RECORDS` valid rows, ordered by `data_id`, are processed. The input database is opened in read-only mode.

In [ ]:
import sqlite3
from pathlib import Path

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Update these paths to match the Kaggle input dataset and model.
INPUT_DB_PATH = Path("/kaggle/input/YOUR-LEGAL-DATABASE/legal.db")
MODEL_DIR = Path("/kaggle/input/YOUR-FINETUNED-MODEL/checkpoint-40236")
OUTPUT_DB_PATH = Path("/kaggle/working/legal_triplet.db")

MAX_RECORDS = 600_000  # Set to 10_000 for a quick debug run.
DATABASE_BATCH_SIZE = 256
EMBEDDING_BATCH_SIZE = 128
INDEX_TRAINING_SAMPLE_SIZE = 100_000
INDEX_NLIST = 1_024
INDEX_NPROBE = 16
SEARCH_TOP_K = 16
PROGRESS_EVERY = 10_000

if not INPUT_DB_PATH.is_file():
    raise FileNotFoundError(f"Input database does not exist: {INPUT_DB_PATH}")
if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Model directory does not exist: {MODEL_DIR}")
if MAX_RECORDS < 1:
    raise ValueError("MAX_RECORDS must be at least 1.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Input DB: {INPUT_DB_PATH}")
print(f"Model: {MODEL_DIR}")
print(f"Output DB: {OUTPUT_DB_PATH}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
model = SentenceTransformer(
    str(MODEL_DIR),
    device=DEVICE,
    local_files_only=True,
)
print(f"Embedding dimension: {model.get_embedding_dimension()}")

In [ ]:
VALID_WHERE = (
    "anchor IS NOT NULL AND positive IS NOT NULL "
    "AND length(trim(anchor)) > 0 AND length(trim(positive)) > 0"
)


def encode_texts(texts: list[str], prefix: str) -> np.ndarray:
    """Encode texts with the E5 query or passage prefix."""
    prefixed_texts = [f"{prefix}: {text}" for text in texts]
    embeddings = model.encode(
        prefixed_texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return np.asarray(embeddings, dtype=np.float32)


def fetch_legal_batch(
    connection: sqlite3.Connection,
    last_data_id: str | None,
    limit: int,
) -> list[dict]:
    """Read one deterministic batch using data_id keyset pagination."""
    query = f"""
        SELECT data_id, source, title, anchor, positive
        FROM legal
        WHERE {VALID_WHERE}
    """
    parameters: list[object] = []
    if last_data_id is not None:
        query += " AND data_id > ?"
        parameters.append(last_data_id)
    query += " ORDER BY data_id LIMIT ?"
    parameters.append(limit)

    rows = connection.execute(query, parameters).fetchall()
    return [dict(row) for row in rows]


def get_source_stats(connection: sqlite3.Connection) -> int:
    """Count the valid rows that are inside the 600k processing limit."""
    row = connection.execute(
        f"""
            SELECT COUNT(*)
            FROM (
                SELECT data_id
                FROM legal
                WHERE {VALID_WHERE}
                ORDER BY data_id
                LIMIT ?
            )
        """,
        (MAX_RECORDS,),
    ).fetchone()
    return int(row[0])


def get_training_texts(
    connection: sqlite3.Connection,
    source_count: int,
) -> list[str]:
    """Take a small sample from the selected rows to train FAISS."""
    sample_size = min(INDEX_TRAINING_SAMPLE_SIZE, source_count)
    rows = connection.execute(
        f"""
            SELECT positive
            FROM (
                SELECT data_id, positive
                FROM legal
                WHERE {VALID_WHERE}
                ORDER BY data_id
                LIMIT ?
            )
            ORDER BY random()
            LIMIT ?
        """,
        (source_count, sample_size),
    ).fetchall()
    texts = [row[0] for row in rows]
    if len(texts) < sample_size:
        raise RuntimeError("Could not read the FAISS training sample.")
    print(f"FAISS training sample: {len(texts):,} passages")
    return texts


def build_index(
    connection: sqlite3.Connection,
    source_count: int,
) -> tuple[faiss.IndexIVFFlat, list[str]]:
    """Build an IVF-Flat cosine-similarity index over selected passages."""
    dimension = model.get_embedding_dimension()
    nlist = min(INDEX_NLIST, max(1, source_count // 100))
    index = faiss.IndexIVFFlat(
        faiss.IndexFlatIP(dimension),
        dimension,
        nlist,
        faiss.METRIC_INNER_PRODUCT,
    )

    training_texts = get_training_texts(connection, source_count)
    training_embeddings = encode_texts(training_texts, "passage")
    print(f"Training FAISS with {len(training_embeddings):,} embeddings...")
    index.train(training_embeddings)
    index.nprobe = min(INDEX_NPROBE, nlist)
    data_ids: list[str] = []
    last_data_id = None
    indexed = 0

    while indexed < source_count:
        batch_limit = min(DATABASE_BATCH_SIZE, source_count - indexed)
        rows = fetch_legal_batch(connection, last_data_id, batch_limit)
        if not rows:
            raise RuntimeError(
                f"Index stopped at {indexed:,} rows; expected {source_count:,}."
            )

        embeddings = encode_texts([row["positive"] for row in rows], "passage")
        if embeddings.shape != (len(rows), dimension):
            raise RuntimeError(f"Unexpected embedding shape: {embeddings.shape}")

        index.add(embeddings)
        data_ids.extend(row["data_id"] for row in rows)
        indexed += len(rows)
        last_data_id = rows[-1]["data_id"]

        if indexed % PROGRESS_EVERY < len(rows) or indexed == source_count:
            print(f"Indexed: {indexed:,}/{source_count:,}")

    if index.ntotal != len(data_ids) or index.ntotal != source_count:
        raise RuntimeError("FAISS index and data_id map have different sizes.")
    return index, data_ids


def get_existing_ids(
    connection: sqlite3.Connection,
    data_ids: list[str],
) -> set[str]:
    if not data_ids:
        return set()

    placeholders = ",".join("?" for _ in data_ids)
    rows = connection.execute(
        f"SELECT data_id FROM legal_triplet WHERE data_id IN ({placeholders})",
        data_ids,
    ).fetchall()
    return {row[0] for row in rows}


def fetch_candidate_positives(
    connection: sqlite3.Connection,
    data_ids: set[str],
) -> dict[str, str]:
    """Fetch only the passages returned by the current FAISS search."""
    positives: dict[str, str] = {}
    values = list(data_ids)

    # Stay below SQLite's usual 999-parameter limit.
    for start in range(0, len(values), 900):
        batch = values[start : start + 900]
        placeholders = ",".join("?" for _ in batch)
        rows = connection.execute(
            f"SELECT data_id, positive FROM legal WHERE data_id IN ({placeholders})",
            batch,
        ).fetchall()
        positives.update(
            {data_id: positive for data_id, positive in rows if positive and positive.strip()}
        )

    return positives


def build_triplets(
    rows: list[dict],
    neighbor_positions: np.ndarray,
    data_ids: list[str],
    candidate_positives: dict[str, str],
) -> tuple[list[tuple], int]:
    triplets: list[tuple] = []
    skipped = 0

    for row, positions in zip(rows, neighbor_positions, strict=True):
        hard_negative = None
        for position in positions:
            if position < 0:
                continue

            candidate_id = data_ids[int(position)]
            candidate_positive = candidate_positives.get(candidate_id)
            if candidate_id == row["data_id"]:
                continue
            if candidate_positive is None or candidate_positive == row["positive"]:
                continue

            hard_negative = candidate_positive
            break

        if hard_negative is None:
            skipped += 1
            continue

        triplets.append(
            (
                row["data_id"],
                row["source"],
                row["title"],
                row["anchor"],
                row["positive"],
                hard_negative,
            )
        )

    return triplets, skipped


def create_output_table(connection: sqlite3.Connection) -> None:
    connection.execute("""
        CREATE TABLE IF NOT EXISTS legal_triplet (
            data_id TEXT PRIMARY KEY,
            source TEXT,
            title TEXT,
            anchor TEXT NOT NULL,
            positive TEXT NOT NULL,
            hard_negative TEXT NOT NULL
        )
    """)
    connection.commit()


In [ ]:
source_connection = sqlite3.connect(
    f"file:{INPUT_DB_PATH.resolve()}?mode=ro",
    uri=True,
)
source_connection.row_factory = sqlite3.Row
output_connection = sqlite3.connect(OUTPUT_DB_PATH)
output_connection.execute("PRAGMA journal_mode = WAL")
output_connection.execute("PRAGMA synchronous = NORMAL")

try:
    columns = {row[1] for row in source_connection.execute("PRAGMA table_info(legal)")}
    required_columns = {"data_id", "source", "title", "anchor", "positive"}
    missing_columns = required_columns.difference(columns)
    if missing_columns:
        raise ValueError(f"Input legal table is missing columns: {sorted(missing_columns)}")

    create_output_table(output_connection)
    source_count = get_source_stats(source_connection)
    if source_count == 0:
        raise ValueError("No valid rows were found in the legal table.")

    total_valid = source_connection.execute(
        f"SELECT COUNT(*) FROM legal WHERE {VALID_WHERE}"
    ).fetchone()[0]
    print(f"Valid source rows: {total_valid:,}")
    print(f"Rows selected for this run: {source_count:,}")

    index, data_ids = build_index(source_connection, source_count)
    print(f"FAISS index ready: {index.ntotal:,} vectors")

    insert_query = """
        INSERT OR IGNORE INTO legal_triplet (
            data_id, source, title, anchor, positive, hard_negative
        ) VALUES (?, ?, ?, ?, ?, ?)
    """

    existing_count = output_connection.execute(
        "SELECT COUNT(*) FROM legal_triplet"
    ).fetchone()[0]
    if existing_count > MAX_RECORDS:
        raise RuntimeError(
            f"Output already has {existing_count:,} rows, above MAX_RECORDS. "
            "Choose a new OUTPUT_DB_PATH."
        )
    print(f"Existing output rows: {existing_count:,}")

    last_data_id = None
    processed = 0
    inserted = 0
    skipped_without_negative = 0

    while processed < source_count:
        batch_limit = min(DATABASE_BATCH_SIZE, source_count - processed)
        rows = fetch_legal_batch(source_connection, last_data_id, batch_limit)
        if not rows:
            raise RuntimeError(
                f"Extraction stopped at {processed:,} rows; expected {source_count:,}."
            )
        last_data_id = rows[-1]["data_id"]

        completed_ids = get_existing_ids(
            output_connection,
            [row["data_id"] for row in rows],
        )
        pending_rows = [
            row for row in rows if row["data_id"] not in completed_ids
        ]

        if pending_rows:
            query_embeddings = encode_texts(
                [row["anchor"] for row in pending_rows],
                "query",
            )
            _, neighbor_positions = index.search(query_embeddings, SEARCH_TOP_K)
            candidate_ids = {
                data_ids[int(position)]
                for positions in neighbor_positions
                for position in positions
                if position >= 0
            }
            candidate_positives = fetch_candidate_positives(
                source_connection,
                candidate_ids,
            )
            triplets, skipped = build_triplets(
                pending_rows,
                neighbor_positions,
                data_ids,
                candidate_positives,
            )
            output_connection.executemany(insert_query, triplets)
            output_connection.commit()

            inserted += len(triplets)
            skipped_without_negative += skipped

        processed += len(rows)
        if processed % PROGRESS_EVERY < len(rows) or processed == source_count:
            print(
                f"Processed: {processed:,}/{source_count:,}; "
                f"inserted this run: {inserted:,}; "
                f"without hard negative: {skipped_without_negative:,}"
            )

    output_count = output_connection.execute(
        "SELECT COUNT(*) FROM legal_triplet"
    ).fetchone()[0]
    integrity = output_connection.execute("PRAGMA integrity_check").fetchone()[0]
    print(f"Finished. Output rows: {output_count:,}; integrity: {integrity}")
finally:
    output_connection.commit()
    output_connection.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    output_connection.execute("PRAGMA journal_mode = DELETE")
    output_connection.close()
    source_connection.close()

print(f"Kaggle output file: {OUTPUT_DB_PATH}")
print(f"Output size: {OUTPUT_DB_PATH.stat().st_size / 1024**3:.2f} GB")